In [0]:
# =============================================================================
# All India Sales Data Pipeline — Medallion Architecture
# Author  : Nandimandalam Mahesh
# Stack   : Azure Databricks · PySpark · Delta Lake · ADLS Gen2
# Layers  : Bronze → Silver → Gold
# Domain  : Consumer Durables — All India Sales Transactions
# =============================================================================
#
# Pipeline Overview:
#   1. BRONZE  — Raw ingestion from simulated source (CSV/JSON)
#   2. SILVER  — Cleansing, validation, deduplication, schema enforcement + derived KPIs
#   3. GOLD    — Aggregated business KPIs ready for Power BI / Tableau reporting
#
# Note: 100% synthetic data only. No real company, IFB, or customer data is present.
#       Perfect for resume / interview showcase — demonstrates real production-grade ETL.
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime
import logging
import random

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIG
# =============================================================================

# In real Azure environment these would be ADLS Gen2 paths:
#   BRONZE_PATH = "abfss://bronze@<storage>.dfs.core.windows.net/india_sales"
# Using /tmp for Databricks Community / demo

BRONZE_PATH  = "/tmp/medallion/bronze/india_sales"
SILVER_PATH  = "/tmp/medallion/silver/india_sales"
GOLD_PATH    = "/tmp/medallion/gold/sales_kpis"
CHECKPOINT   = "/tmp/medallion/checkpoints"

PIPELINE_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# =============================================================================
# SPARK SESSION
# =============================================================================

spark = (
    SparkSession.builder
    .appName("All_India_Sales_Medallion_Pipeline")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.optimizeWrite.enabled", "true")
    .config("spark.databricks.delta.autoCompact.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# =============================================================================
# SYNTHETIC DATA GENERATOR
# (Simulates daily sales file from All-India dealer network / ERP system)
# =============================================================================

def generate_synthetic_batch(num_records: int = 800):
    """
    Generates realistic All-India sales data.
    Intentionally includes ~8% dirty records for Silver-layer DQ demonstration.
    Covers 10 major states + zonal distribution.
    """
    states = [
        "Maharashtra", "Karnataka", "Tamil Nadu", "Delhi", "Gujarat",
        "Uttar Pradesh", "West Bengal", "Telangana", "Rajasthan", "Kerala"
    ]
    regions = ["North", "South", "East", "West", "Central"]
    categories = ["Laundry", "Kitchen", "Cooling", "Dishwasher"]
    products = {
        "Laundry": ["WM-101", "WM-201", "DRYER-301"],
        "Kitchen": ["RF-401", "MW-501", "DW-601"],
        "Cooling": ["AC-701", "AC-801"],
        "Dishwasher": ["DW-901"]
    }
    dealers = [f"DLR-{str(i).zfill(4)}" for i in range(1, 51)]
    payment_types = ["Cash", "Card", "UPI", "EMI", "Finance"]

    base_time = datetime(2025, 1, 1)
    records = []

    for i in range(num_records):
        sale_date = base_time + timedelta(days=random.randint(0, 365 * 1))
        category = random.choice(categories)
        product_code = random.choice(products[category])

        # Inject ~8% dirty records
        dirty = random.random() < 0.08

        record = {
            "sales_id":       f"SLS-{str(i + 1).zfill(7)}" if not dirty else None,
            "invoice_id":     f"INV-{str(i + 1).zfill(7)}",
            "state":          random.choice(states),
            "region":         random.choice(regions),
            "product_category": category,
            "product_code":   product_code,
            "dealer_id":      random.choice(dealers),
            "quantity_sold":  random.randint(1, 15) if not dirty else -2,
            "unit_price":     round(random.uniform(18000, 145000), 2),
            "discount_pct":   round(random.uniform(0, 25), 2) if not dirty else 120.0,  # invalid >100
            "payment_type":   random.choice(payment_types),
            "sale_date":      sale_date.strftime("%Y-%m-%d %H:%M:%S"),
            "load_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

        records.append(record)
        # ~3% duplicates
        if random.random() < 0.03:
            records.append(record)

    return records


# =============================================================================
# SCHEMAS
# =============================================================================

RAW_SCHEMA = StructType([
    StructField("sales_id",          StringType(),  True),
    StructField("invoice_id",        StringType(),  True),
    StructField("state",             StringType(),  True),
    StructField("region",            StringType(),  True),
    StructField("product_category",  StringType(),  True),
    StructField("product_code",      StringType(),  True),
    StructField("dealer_id",         StringType(),  True),
    StructField("quantity_sold",     IntegerType(), True),
    StructField("unit_price",        DoubleType(),  True),
    StructField("discount_pct",      DoubleType(),  True),
    StructField("payment_type",      StringType(),  True),
    StructField("sale_date",         StringType(),  True),
    StructField("load_timestamp",    StringType(),  True),
])


# =============================================================================
# LAYER 1 — BRONZE (Raw Ingestion)
# =============================================================================

def ingest_to_bronze():
    """Simulates landing raw sales file from All-India dealer network."""
    logger.info("[ BRONZE ] Starting raw ingestion ...")

    raw_records = generate_synthetic_batch(num_records=800)
    raw_df = spark.createDataFrame(raw_records, schema=RAW_SCHEMA)

    # Add pipeline metadata
    raw_df = raw_df.withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID)) \
                   .withColumn("ingestion_ts", F.current_timestamp()) \
                   .withColumn("source_file",  F.lit("erp_sales_export_daily.csv"))

    record_count = raw_df.count()
    logger.info(f"[ BRONZE ] Records received: {record_count:,}")

    raw_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save(BRONZE_PATH)

    logger.info(f"[ BRONZE ] Written to: {BRONZE_PATH}")
    return record_count


# =============================================================================
# LAYER 2 — SILVER (Cleansing, Validation, Deduplication)
# =============================================================================

def transform_to_silver():
    """
    Applies business DQ rules and upserts into Silver using Delta MERGE.
    Rules:
      1. Drop null sales_id
      2. Drop invalid quantity (<= 0)
      3. Drop invalid discount (> 100 or < 0)
      4. Deduplicate on sales_id (keep latest)
      5. Cast sale_date + derive net_revenue & month_year
    """
    logger.info("[ SILVER ] Starting cleansing & transformation ...")

    bronze_df = spark.read.format("delta").load(BRONZE_PATH)

    before = bronze_df.count()

    # Rule 1: Critical key nulls
    bronze_df = bronze_df.filter(F.col("sales_id").isNotNull())
    after_nulls = bronze_df.count()
    logger.info(f"[ SILVER ] Null sales_id dropped: {before - after_nulls:,} records")

    # Rule 2 & 3: Business validation
    bronze_df = bronze_df.filter(
        (F.col("quantity_sold") > 0) &
        (F.col("discount_pct") >= 0) &
        (F.col("discount_pct") <= 100)
    )
    after_invalid = bronze_df.count()
    logger.info(f"[ SILVER ] Invalid qty/discount dropped: {after_nulls - after_invalid:,} records")

    # Rule 4: Deduplication (keep latest ingestion)
    from pyspark.sql.window import Window
    window = Window.partitionBy("sales_id").orderBy(F.col("ingestion_ts").desc())
    clean_df = bronze_df.withColumn("row_num", F.row_number().over(window)) \
                        .filter(F.col("row_num") == 1) \
                        .drop("row_num")

    dupes_removed = after_invalid - clean_df.count()
    logger.info(f"[ SILVER ] Duplicates removed: {dupes_removed:,} records")

    # Rule 5: Type casting + business metrics
    clean_df = clean_df.withColumn(
        "sale_date", F.to_timestamp("sale_date", "yyyy-MM-dd HH:mm:ss")
    ).withColumn(
        "month_year", F.date_format(F.col("sale_date"), "yyyy-MM")
    ).withColumn(
        "net_revenue",
        F.round(
            F.col("quantity_sold") * F.col("unit_price") * (1 - F.col("discount_pct") / 100),
            2
        )
    ).withColumn(
        "gross_revenue",
        F.round(F.col("quantity_sold") * F.col("unit_price"), 2)
    ).withColumn(
        "silver_processed_ts", F.current_timestamp()
    )

    # MERGE (SCD Type 1) into Silver
    if DeltaTable.isDeltaTable(spark, SILVER_PATH):
        silver_table = DeltaTable.forPath(spark, SILVER_PATH)
        silver_table.alias("target").merge(
            clean_df.alias("source"),
            "target.sales_id = source.sales_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        logger.info("[ SILVER ] Delta MERGE (upsert) completed.")
    else:
        clean_df.write.format("delta").mode("overwrite") \
            .partitionBy("region", "month_year") \
            .save(SILVER_PATH)
        logger.info("[ SILVER ] Initial write completed (partitioned by region + month).")

    final_count = spark.read.format("delta").load(SILVER_PATH).count()
    logger.info(f"[ SILVER ] Total records in Silver table: {final_count:,}")
    return final_count


# =============================================================================
# LAYER 3 — GOLD (Business KPIs for Reporting)
# =============================================================================

def aggregate_to_gold():
    """
    Creates two Gold tables:
      1. State & Zonal Sales Performance (monthly)
      2. Product Category Performance (All-India)
    """
    logger.info("[ GOLD ] Starting aggregation ...")

    silver_df = spark.read.format("delta").load(SILVER_PATH)

    # ── Gold Table 1: Monthly State & Zonal KPIs ──
    state_kpis = silver_df \
        .withColumn("sale_month", F.to_date(F.col("sale_date"), "yyyy-MM-dd")) \
        .groupBy("sale_month", "state", "region", "product_category") \
        .agg(
            F.count("sales_id").alias("total_transactions"),
            F.sum("quantity_sold").alias("total_units_sold"),
            F.sum("gross_revenue").alias("gross_revenue_inr"),
            F.sum("net_revenue").alias("net_revenue_inr"),
            F.round(F.avg("discount_pct"), 2).alias("avg_discount_pct"),
            F.countDistinct("dealer_id").alias("unique_dealers"),
        ) \
        .withColumn("gold_refreshed_ts", F.current_timestamp())

    state_kpis.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("sale_month") \
        .save(GOLD_PATH + "/state_zonal_kpis")

    logger.info(f"[ GOLD ] state_zonal_kpis rows: {state_kpis.count():,}")

    # ── Gold Table 2: Product Category Performance ──
    product_kpis = silver_df \
        .groupBy("product_category", "product_code") \
        .agg(
            F.count("sales_id").alias("total_transactions"),
            F.sum("quantity_sold").alias("total_units"),
            F.sum("net_revenue").alias("total_net_revenue_inr"),
            F.round(F.avg("discount_pct"), 2).alias("avg_discount_pct"),
            F.round(F.avg("unit_price"), 2).alias("avg_unit_price_inr"),
            F.countDistinct("state").alias("states_sold_in"),
        ) \
        .withColumn(
            "revenue_share_pct",
            F.round(
                F.col("total_net_revenue_inr") /
                F.sum("total_net_revenue_inr").over() * 100, 2
            )
        ) \
        .withColumn("gold_refreshed_ts", F.current_timestamp())

    product_kpis.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(GOLD_PATH + "/product_category_kpis")

    logger.info(f"[ GOLD ] product_category_kpis rows: {product_kpis.count():,}")

    # Preview for notebook visibility
    logger.info("[ GOLD ] === State & Zonal KPIs Preview (Top 10) ===")
    state_kpis.orderBy(F.col("net_revenue_inr").desc()).show(10, truncate=False)

    logger.info("[ GOLD ] === Product Category Performance Preview ===")
    product_kpis.orderBy(F.col("total_net_revenue_inr").desc()).show(truncate=False)


# =============================================================================
# DATA QUALITY REPORT
# =============================================================================

def generate_dq_report():
    bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
    silver_count = spark.read.format("delta").load(SILVER_PATH).count()
    rejection_rate = round((bronze_count - silver_count) / bronze_count * 100, 2) if bronze_count > 0 else 0

    print("\n" + "=" * 65)
    print("       DATA QUALITY REPORT — All India Sales Pipeline")
    print("=" * 65)
    print(f"  Pipeline Run ID      : {PIPELINE_RUN_ID}")
    print(f"  Bronze Records       : {bronze_count:,}")
    print(f"  Silver Records       : {silver_count:,}")
    print(f"  Records Rejected     : {bronze_count - silver_count:,}")
    print(f"  Rejection Rate       : {rejection_rate}%")
    print(f"  DQ Status            : {'PASS ✓' if rejection_rate < 12 else 'REVIEW ⚠'}")
    print("=" * 65 + "\n")


# =============================================================================
# DELTA LAKE OPTIMISATION
# =============================================================================

def optimise_delta_tables():
    logger.info("[ OPTIMISE ] Running OPTIMIZE + ZORDER on Silver table ...")
    spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}` ZORDER BY (region, product_category, month_year)")
    logger.info("[ OPTIMISE ] Done.")


# =============================================================================
# MAIN PIPELINE ORCHESTRATION
# =============================================================================

def run_pipeline():
    logger.info(f"{'=' * 70}")
    logger.info(f"  ALL INDIA SALES PIPELINE START — Run ID: {PIPELINE_RUN_ID}")
    logger.info(f"{'=' * 70}")

    try:
        bronze_count = ingest_to_bronze()
        silver_count = transform_to_silver()
        aggregate_to_gold()
        generate_dq_report()
        # optimise_delta_tables()   # Uncomment in paid cluster

        logger.info("[ PIPELINE ] ✅ Completed successfully — Ready for BI dashboards!")

    except Exception as e:
        logger.error(f"[ PIPELINE ] ❌ FAILED — {str(e)}")
        raise


if __name__ == "__main__":
    run_pipeline()

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-6458666280174073>, line 61
     47 # =============================================================================
     48 # SPARK SESSION
     49 # =============================================================================
     51 spark = (
     52     SparkSession.builder
     53     .appName("All_India_Sales_Medallion_Pipeline")
   (...)
     58     .getOrCreate()
     59 )
---> 61 spark.sparkContext.setLogLevel("WARN")
     63 # =============================================================================
     64 # SYNTHETIC DATA GENERATOR
     65 # (Simulates daily sales file from All-India dealer network / ERP system)
     66 # =============================================================================
     68 def generate_synthetic_batch(num_records: int = 800):

File /databricks/python/lib/python3.12/site-pack